# Fetch New Agent PRs and Build Updated Dataset

This notebook:
1. Searches GitHub for **recent merged PRs authored by AI agents** (Claude Code, Copilot, Cursor, Devin, OpenAI Codex) using the same five-agent taxonomy as the original study.
2. Processes each PR through the exact same mining pipeline defined in `build.py` (Lizard metrics, Semgrep, documentation extraction, turnover).
3. Saves the result to `dataset/data/new_agent_dataset.csv` with the same 30-column schema as `final_dataset.csv`.
4. Merges `final_dataset.csv` + `new_agent_dataset.csv` → `dataset/data/final_dataset_new.csv`.

**Prerequisites:**
- A `.env` file in the project root with at least `GITHUB_TOKEN_1` (and optionally `GITHUB_TOKEN_2`, `GITHUB_TOKEN_3` for parallel processing).
- `lizard`, `semgrep`, `git` available on PATH.
- `pip install python-dotenv requests pandas tqdm textstat`

In [ ]:
# ── 0. Environment must be loaded BEFORE importing build.py (which checks tokens) ──
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Try several .env locations (notebook may be run from different CWDs)
for env_candidate in [
    Path('.') / '.env',
    Path('..') / '.env',
    Path('../..') / '.env',
]:
    if env_candidate.exists():
        load_dotenv(env_candidate)
        print(f'Loaded .env from {env_candidate.resolve()}')
        break

missing = [k for k in ['GITHUB_TOKEN_1'] if not os.environ.get(k)]
if missing:
    raise EnvironmentError(
        f"Missing required environment variables: {missing}\n"
        "Create a .env file in the project root with GITHUB_TOKEN_1=ghp_..."
    )
print('GitHub token(s) detected:', [k for k in ['GITHUB_TOKEN_1','GITHUB_TOKEN_2','GITHUB_TOKEN_3'] if os.environ.get(k)])

In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import re
import time
import json
import math
import csv
import shutil
import tempfile
import subprocess
import textwrap
import warnings
import numpy as np
import pandas as pd
import requests
import tqdm
from collections import Counter
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Any

warnings.filterwarnings('ignore')

# ── Paths (works whether CWD is project root or this folder) ──────────────────
THIS_DIR = Path(os.path.abspath(''))
BUILD_DIR = THIS_DIR if (THIS_DIR / 'build.py').exists() else THIS_DIR / 'dataset' / 'buildDataset'
DATA_DIR  = BUILD_DIR.parent / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

FINAL_DATASET_PATH    = DATA_DIR / 'final_dataset.csv'
NEW_AGENT_OUTPUT_PATH = DATA_DIR / 'new_agent_dataset.csv'
MERGED_OUTPUT_PATH    = DATA_DIR / 'final_dataset_new.csv'
PR_LIST_PATH          = DATA_DIR / 'new_agent_pr_list.parquet'

print('Build dir:', BUILD_DIR)
print('Data dir: ', DATA_DIR)
print('final_dataset.csv exists:', FINAL_DATASET_PATH.exists())

In [ ]:
# ── 2. Import the mining pipeline from build.py ───────────────────────────────
# build.py must be importable — env tokens are already set above.
sys.path.insert(0, str(BUILD_DIR))

try:
    from build import (
        tokenize, calculate_entropy, doc_redundancy, doc_code_overlap,
        strip_comments, find_documentation_header, extract_documentation,
        parse_detailed_lizard, find_file_at_commit,
        get_time_based_shas, get_target_shas, getTurnover,
        AiDevMiner, REPO_BASE_DIR, SUPPORTED_EXTENSIONS, TOKENS,
    )
    print('Successfully imported pipeline from build.py')
    print(f'  REPO_BASE_DIR:        {REPO_BASE_DIR}')
    print(f'  SUPPORTED_EXTENSIONS: {len(SUPPORTED_EXTENSIONS)} types')
    print(f'  Active tokens:        {len(TOKENS)}')
except ImportError as e:
    raise ImportError(
        f'Could not import from build.py: {e}\n'
        f'Make sure {BUILD_DIR}/build.py exists and all its dependencies are installed.'
    )

## Step 1 — Fetch recent agent PRs from GitHub

We use the GitHub Search API with per-agent query signatures to find recently merged PRs authored by each AI agent. Adjust `TARGET_PRS_PER_AGENT` and `DATE_AFTER` to control the fetch size.

In [ ]:
# ── 3. Fetch configuration ────────────────────────────────────────────────────

# How many PRs to collect per agent (set lower for testing, e.g. 10)
TARGET_PRS_PER_AGENT = 200

# Only fetch PRs merged after this date (ISO format).
# Set to a date after your existing final_dataset was collected to avoid overlap.
DATE_AFTER = '2025-01-01'

# Skip repositories larger than this (KB) — avoids giant monorepos
MAX_REPO_SIZE_KB = 500_000

# GitHub Search API — uses ONE primary token (rate limit: 30 requests/min)
SEARCH_TOKEN = os.environ.get('GITHUB_TOKEN_1') or TOKENS[0]
SEARCH_HEADERS = {
    'Authorization': f'token {SEARCH_TOKEN}',
    'Accept': 'application/vnd.github.v3+json',
}

# ── Per-agent GitHub search queries ──────────────────────────────────────────
# Each tuple: (agent_label_matching_final_dataset, github_search_query)
AGENT_QUERIES = [
    (
        'Claude_Code',
        # Claude Code leaves a Co-Authored-By trailer with the anthropic noreply address
        f'is:pr is:merged merged:>{DATE_AFTER} "noreply@anthropic.com" in:body',
    ),
    (
        'Copilot',
        # GitHub Copilot Workspace PRs are authored by the copilot[bot] user
        f'is:pr is:merged merged:>{DATE_AFTER} author:copilot[bot]',
    ),
    (
        'Cursor',
        # Cursor agent signs commits with cursor-noreply@cursor.sh
        f'is:pr is:merged merged:>{DATE_AFTER} "cursor-noreply@cursor.sh" in:body',
    ),
    (
        'Devin',
        # Devin PRs are authored by the devin-ai-integration[bot] GitHub app
        f'is:pr is:merged merged:>{DATE_AFTER} author:devin-ai-integration[bot]',
    ),
    (
        'OpenAI_Codex',
        # OpenAI Codex agent links to chatgpt.com/codex in the PR body
        f'is:pr is:merged merged:>{DATE_AFTER} "chatgpt.com/codex" in:body',
    ),
]

print('Agent search configuration:')
for agent, q in AGENT_QUERIES:
    print(f'  {agent:<16} -> {q}')

In [ ]:
# ── 4. GitHub PR fetcher ───────────────────────────────────────────────────────

def get_repo_size_kb(repo_api_url: str) -> int:
    """Return repo size in KB via GitHub API."""
    try:
        r = requests.get(repo_api_url, headers=SEARCH_HEADERS, timeout=10)
        if r.status_code == 200:
            return r.json().get('size', 0)
    except Exception:
        pass
    return 0

def fetch_agent_prs(agent_label: str, query: str, target: int) -> List[dict]:
    """
    Fetch up to `target` merged PRs matching `query` from the GitHub Search API.
    Returns a list of PR dicts in the same schema as all_pull_request_ballanced.parquet.
    """
    prs = []
    page = 1
    per_page = 100
    seen_ids = set()

    while len(prs) < target:
        url = 'https://api.github.com/search/issues'
        params = {
            'q': query, 'per_page': per_page, 'page': page,
            'sort': 'created', 'order': 'desc'
        }

        try:
            r = requests.get(url, headers=SEARCH_HEADERS, params=params, timeout=20)
        except requests.RequestException as e:
            print(f'  Request error on page {page}: {e}')
            break

        # Handle rate limiting
        if r.status_code in (403, 429):
            reset_at = int(r.headers.get('X-RateLimit-Reset', time.time() + 60))
            wait = max(reset_at - int(time.time()), 0) + 2
            print(f'  Rate limited. Waiting {wait}s...')
            time.sleep(wait)
            continue

        if r.status_code != 200:
            print(f'  Search API error {r.status_code}: {r.text[:200]}')
            break

        data  = r.json()
        items = data.get('items', [])
        if not items:
            break

        for item in items:
            if len(prs) >= target:
                break

            item_id = item.get('id')
            if item_id in seen_ids:
                continue
            seen_ids.add(item_id)

            repo_api_url = item.get('repository_url', '')

            # Skip oversized repos
            if get_repo_size_kb(repo_api_url) > MAX_REPO_SIZE_KB:
                continue

            merged_at = item.get('pull_request', {}).get('merged_at')
            if not merged_at:
                continue  # search occasionally returns unmerged items

            prs.append({
                'id':         item_id,
                'number':     item.get('number'),
                'title':      item.get('title'),
                'body':       item.get('body'),
                'agent':      agent_label,
                'user_id':    item.get('user', {}).get('login'),
                'user':       item.get('user', {}).get('login'),
                'state':      item.get('state'),
                'created_at': item.get('created_at'),
                'closed_at':  item.get('closed_at'),
                'merged_at':  merged_at,
                'repo_id':    None,
                # Keep API URL format — this is what build.py expects in row['repo_url']
                'repo_url':   repo_api_url,
                'html_url':   item.get('html_url'),
            })

        # Respect the GitHub search rate limit (30 requests per minute)
        time.sleep(2.2)
        page += 1

        # GitHub search caps at 1000 results per query
        total_available = data.get('total_count', 0)
        if page * per_page > min(1000, total_available):
            break

    return prs

In [ ]:
# ── 5. Run the fetch for all agents ───────────────────────────────────────────
all_new_prs = []

for agent_label, query in AGENT_QUERIES:
    print(f'\nFetching {agent_label} PRs...')
    prs = fetch_agent_prs(agent_label, query, TARGET_PRS_PER_AGENT)
    print(f'  -> {len(prs)} PRs collected')
    all_new_prs.extend(prs)

df_prs = pd.DataFrame(all_new_prs)
print(f'\nTotal new PRs fetched: {len(df_prs)}')
print(df_prs['agent'].value_counts().to_string())

In [ ]:
# ── 6. Deduplicate against final_dataset.csv to avoid overlap ─────────────────
if FINAL_DATASET_PATH.exists():
    df_existing = pd.read_csv(FINAL_DATASET_PATH, usecols=['repo', 'pull_request'])
    # final_dataset repo column: https://api.github.com/repos/...
    # df_prs repo_url column:    https://api.github.com/repos/...
    existing_keys = set(
        zip(df_existing['repo'].astype(str), df_existing['pull_request'].astype(str))
    )

    before = len(df_prs)
    df_prs = df_prs[
        ~df_prs.apply(
            lambda row: (str(row['repo_url']), str(row['number'])) in existing_keys,
            axis=1
        )
    ].reset_index(drop=True)

    print(f'Removed {before - len(df_prs)} PRs already in final_dataset.csv')
else:
    print('final_dataset.csv not found — processing all fetched PRs')

print(f'PRs to process: {len(df_prs)}')

# Save PR list for reproducibility
df_prs.to_parquet(PR_LIST_PATH, index=False)
print(f'PR list saved to {PR_LIST_PATH}')

## Step 2 — Mine each PR through the build pipeline

This uses the same `AiDevMiner.process_pr()` from `build.py`. It clones each repository, runs Lizard for code metrics, Semgrep for quality checks, extracts documentation, and computes turnover at commit offsets C5/C10/C20 and months M1/M3.

**Expected runtime:** ~2–5 minutes per PR (dominated by git clone + Semgrep). With `TARGET_PRS_PER_AGENT=200` and 5 agents this can take several hours — reduce `TARGET_PRS_PER_AGENT` to `10` for a quick smoke test.

In [ ]:
# ── 7. Run the mining pipeline ─────────────────────────────────────────────────
# Column order exactly matches final_dataset.csv
FINAL_COLUMNS = [
    'repo', 'pull_request', 'label', 'file_path', 'function_name',
    'function_start_line', 'function_end_line', 'function',
    'loc', 'sloc', 'cyclomatic_complexity', 'num_parameters',
    'doc_lines', 'doc_text', 'doc_entropy', 'total_entropy',
    'doc_readability', 'semgrep_findings', 'semgrep_findings_count',
    'doc_code_overlap', 'doc_redundancy',
    'pr_date_merged', 'pr_date_created', 'pr_date_closed',
    'turnover_c5', 'turnover_c10', 'turnover_c20',
    'turnover_m1', 'turnover_m3',
    'group',  # added post-process: always 'agent' for this dataset
]

miner      = AiDevMiner()
MAX_WORKERS = max(1, len(TOKENS))
final_stats = Counter()

with open(NEW_AGENT_OUTPUT_PATH, 'w', encoding='utf-8', newline='') as f:
    writer = None

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_pr = {
            executor.submit(miner.process_pr, row, TOKENS[i % len(TOKENS)]): row.get('number', '?')
            for i, (_, row) in enumerate(df_prs.iterrows())
        }

        for future in tqdm.tqdm(
            as_completed(future_to_pr),
            total=len(future_to_pr),
            desc='Mining agent PRs'
        ):
            try:
                result_pkg   = future.result()
                results_list = result_pkg['data']
                final_stats.update(result_pkg['stats'])

                for row_result in results_list:
                    row_result['group'] = 'agent'  # always 'agent' for this dataset

                    if writer is None:
                        writer = csv.DictWriter(f, fieldnames=FINAL_COLUMNS)
                        writer.writeheader()

                    writer.writerow({k: row_result.get(k) for k in FINAL_COLUMNS})

            except Exception as e:
                print(f'\n[ERROR] PR {future_to_pr[future]}: {e}')

print('\nMining complete.')
for k, v in sorted(final_stats.items()):
    print(f'  {k}: {v}')

stats_path = DATA_DIR / 'mining_stats_new_agent.json'
with open(stats_path, 'w') as sj:
    json.dump(dict(final_stats), sj, indent=4)
print(f'Stats saved to {stats_path}')

In [ ]:
# ── 8. Inspect new_agent_dataset.csv ──────────────────────────────────────────
df_new = pd.read_csv(NEW_AGENT_OUTPUT_PATH)

print(f'new_agent_dataset.csv shape: {df_new.shape}')
print(f'Columns match final_dataset: {list(df_new.columns) == FINAL_COLUMNS}')
print()
print('Functions per agent label:')
print(df_new['label'].value_counts().to_string())
print()
print('Documentation rate (doc_lines > 0):')
for lbl in df_new['label'].unique():
    sub  = df_new[df_new['label'] == lbl]
    rate = (pd.to_numeric(sub['doc_lines'], errors='coerce') > 0).mean()
    print(f'  {lbl}: {rate:.1%}  (n={len(sub)})')
print()
display(df_new[['label','repo','pull_request','function_name','doc_lines','sloc','cyclomatic_complexity']].head())

## Step 3 — Merge with final_dataset.csv

In [ ]:
# ── 9. Merge new_agent_dataset.csv + final_dataset.csv → final_dataset_new.csv ──

if not FINAL_DATASET_PATH.exists():
    raise FileNotFoundError(f'{FINAL_DATASET_PATH} not found. Cannot merge.')

df_original = pd.read_csv(FINAL_DATASET_PATH)
df_new      = pd.read_csv(NEW_AGENT_OUTPUT_PATH)

print(f'final_dataset.csv:     {df_original.shape}')
print(f'new_agent_dataset.csv: {df_new.shape}')
print()
print('Group breakdown in original dataset:')
print(df_original['group'].value_counts().to_string())

# Align columns before concat (handles any ordering differences)
df_new_aligned = df_new.reindex(columns=df_original.columns)

df_merged = pd.concat([df_original, df_new_aligned], ignore_index=True)

# Deduplicate on the natural key — guards against accidental overlap
before    = len(df_merged)
df_merged = df_merged.drop_duplicates(
    subset=['repo', 'pull_request', 'function_name', 'function_start_line'],
    keep='first'
).reset_index(drop=True)

if before - len(df_merged):
    print(f'Dropped {before - len(df_merged)} duplicate rows.')

print()
print(f'Merged dataset shape: {df_merged.shape}')
print('Group breakdown in merged dataset:')
print(df_merged['group'].value_counts().to_string())
print('Label breakdown in merged dataset:')
print(df_merged['label'].value_counts().to_string())

df_merged.to_csv(MERGED_OUTPUT_PATH, index=False)
print(f'\nSaved to: {MERGED_OUTPUT_PATH}')

In [ ]:
# ── 10. Sanity checks ─────────────────────────────────────────────────────────
df_check = pd.read_csv(MERGED_OUTPUT_PATH)

assert list(df_check.columns) == list(df_original.columns), 'Column mismatch!'
assert len(df_check) >= len(df_original), 'Merged file is smaller than original!'

print('All sanity checks passed.')
print(f'  Columns:        {len(df_check.columns)} (matches original)')
print(f'  Total rows:     {len(df_check):,}')
print(f'  Original rows:  {len(df_original):,}')
print(f'  New rows added: {len(df_check) - len(df_original):,}')
print()
for grp in ['agent', 'human']:
    sub = df_check[df_check['group'] == grp]
    if len(sub):
        doc_rate = (pd.to_numeric(sub['doc_lines'], errors='coerce') > 0).mean()
        print(f'  [{grp}]  n={len(sub):,}  doc_rate={doc_rate:.1%}')

print(f'\nfinal_dataset_new.csv ready at:\n  {MERGED_OUTPUT_PATH}')